# Comparação de desempenho por mediana e intervalo interquartil

Este notebook redesenha as comparações de IGD e HV sem recalcular as métricas. Os resultados existentes são convertidos para formato longo; em seguida, cada combinação de cenário e método é resumida pela mediana e pelo intervalo interquartil das dez repetições. O NBI original é incluído apenas nos painéis com $m=4$, nos quais ele é metodologicamente aplicável.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
SOURCE_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_metrics.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'performance_interval_plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_WIDTH_CM = 16.0
CM_TO_INCH = 1 / 2.54

METHOD_ORDER = ['C-NBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D', 'NBI original']
METHOD_COLORS = {
    'C-NBI': '#E69F00',
    'VRF-NBI': '#0072B2',
    'NSGA-III': '#009E73',
    'MOEA/D': '#D62728',
    'NBI original': '#7B3294',
}
METHOD_MARKERS = {
    'C-NBI': 'o',
    'VRF-NBI': '^',
    'NSGA-III': 's',
    'MOEA/D': 'D',
    'NBI original': 'P',
}
M_VALUES = [4, 6, 12]
CORRELATION_ORDER = ['baixa', 'média', 'alta']
VERSION_ORDER = ['complete', 'equalized']

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.0,
    'axes.titlesize': 10.0,
    'axes.labelsize': 9.5,
    'xtick.labelsize': 8.0,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 7.4,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Fonte:', SOURCE_PATH.relative_to(ROOT))
print('Saída:', OUT_DIR.relative_to(ROOT))

In [ ]:
raw = pd.read_csv(SOURCE_PATH)
selected = raw.loc[
    raw['method'].isin(['CNBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D', 'NBI'])
    & raw['comparison'].isin(['complete', 'equal_cardinality'])
    , ['scenario', 'seed', 'method', 'comparison', 'IGD', 'HV']
].copy()
selected['m'] = selected['scenario'].str.extract(r'^m(\d+)_')[0].astype(int)
selected['correlation'] = selected['scenario'].str.extract(r'^m\d+_(.+)$')[0]
selected['correlation'] = selected['correlation'].map(
    {'low': 'baixa', 'medium': 'média', 'high': 'alta'}
)
selected['method'] = selected['method'].map({'CNBI': 'C-NBI', 'NBI': 'NBI original'}).fillna(selected['method'])
selected['version'] = selected['comparison'].map(
    {'complete': 'complete', 'equal_cardinality': 'equalized'}
)

long_data = selected.melt(
    id_vars=['scenario', 'm', 'correlation', 'method', 'seed', 'version'],
    value_vars=['IGD', 'HV'],
    var_name='metric', value_name='value',
)
long_data['correlation'] = pd.Categorical(
    long_data['correlation'], categories=CORRELATION_ORDER, ordered=True
)
long_data['method'] = pd.Categorical(
    long_data['method'], categories=METHOD_ORDER, ordered=True
)

assert long_data['value'].notna().all()
assert set(long_data['m']) == set(M_VALUES)
assert set(long_data['version']) == set(VERSION_ORDER)
assert set(long_data['metric']) == {'IGD', 'HV'}
nbi_rows = long_data.loc[long_data['method'].eq('NBI original')]
assert not nbi_rows.empty and nbi_rows['m'].eq(4).all()
assert not long_data.loc[long_data['m'].isin([6, 12]), 'method'].eq('NBI original').any()
group_sizes = long_data.groupby(
    ['version', 'metric', 'scenario', 'method'], observed=True
).size()
assert group_sizes.eq(10).all()

long_path = OUT_DIR / 'igd_hv_long_format.csv'
long_data.to_csv(long_path, index=False)
print('Observações em formato longo:', len(long_data))
print('Grupos com dez repetições:', len(group_sizes))

In [ ]:
summary = (
    long_data.groupby(
        ['version', 'metric', 'scenario', 'm', 'correlation', 'method'],
        observed=True, sort=False,
    )['value']
    .agg(
        n='size',
        q1=lambda values: values.quantile(0.25),
        median='median',
        q3=lambda values: values.quantile(0.75),
    )
    .reset_index()
)
summary['correlation'] = pd.Categorical(
    summary['correlation'], categories=CORRELATION_ORDER, ordered=True
)
summary['method'] = pd.Categorical(
    summary['method'], categories=METHOD_ORDER, ordered=True
)
summary = summary.sort_values(
    ['version', 'metric', 'm', 'correlation', 'method']
).reset_index(drop=True)
assert len(summary) == 156
assert summary['n'].eq(10).all()
assert (summary['q1'] <= summary['median']).all()
assert (summary['median'] <= summary['q3']).all()
summary_path = OUT_DIR / 'igd_hv_median_iqr_summary.csv'
summary.to_csv(summary_path, index=False)
summary.head()

In [ ]:
CORRELATION_POSITIONS = dict(zip(CORRELATION_ORDER, np.arange(3, dtype=float)))

def methods_for_panel(m):
    return METHOD_ORDER if m == 4 else METHOD_ORDER[:-1]

def centered_offsets(methods, total_span=0.52):
    values = np.linspace(-total_span / 2, total_span / 2, len(methods))
    return dict(zip(methods, values))

def padded_limits(panel_summary):
    lower = float(panel_summary['q1'].min())
    upper = float(panel_summary['q3'].max())
    span = upper - lower
    if span <= 0:
        span = max(abs(upper), 1.0) * 0.10
    pad = 0.13 * span
    return max(0.0, lower - pad), upper + pad

def plot_median_iqr(data_long, metric, version, output_stem):
    figure, axes = plt.subplots(
        1, 3, sharey=False,
        figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 7.0 * CM_TO_INCH),
    )
    figure.subplots_adjust(
        left=0.085, right=0.99, top=0.88, bottom=0.27, wspace=0.30
    )
    metric_data = data_long.loc[
        data_long['metric'].eq(metric) & data_long['version'].eq(version)
    ]
    metric_summary = (
        metric_data.groupby(
            ['scenario', 'm', 'correlation', 'method'],
            observed=True, sort=False,
        )['value']
        .agg(
            n='size',
            q1=lambda values: values.quantile(0.25),
            median='median',
            q3=lambda values: values.quantile(0.75),
        )
        .reset_index()
    )
    assert metric_summary['n'].eq(10).all()

    for panel_index, (axis, m) in enumerate(zip(axes, M_VALUES)):
        panel = metric_summary.loc[metric_summary['m'].eq(m)].copy()
        panel_methods = methods_for_panel(m)
        offsets = centered_offsets(panel_methods)

        for correlation in CORRELATION_ORDER:
            center = CORRELATION_POSITIONS[correlation]
            for method in panel_methods:
                row = panel.loc[
                    panel['correlation'].eq(correlation)
                    & panel['method'].eq(method)
                ]
                assert len(row) == 1
                row = row.iloc[0]
                position = center + offsets[method]
                median = float(row['median'])
                lower_error = median - float(row['q1'])
                upper_error = float(row['q3']) - median
                marker_size = 6.0 if method == 'C-NBI' else 5.2
                axis.errorbar(
                    position, median,
                    yerr=np.array([[lower_error], [upper_error]]),
                    fmt=METHOD_MARKERS[method], linestyle='none',
                    color=METHOD_COLORS[method],
                    markerfacecolor=METHOD_COLORS[method],
                    markeredgecolor='white', markeredgewidth=0.45,
                    markersize=marker_size, elinewidth=1.25,
                    capsize=3.0, capthick=1.05, zorder=3,
                )

        panel_letter = chr(ord('a') + panel_index)
        axis.set_title(rf'({panel_letter}) $m={m}$', pad=5)
        axis.set_xticks(np.arange(3), CORRELATION_ORDER)
        axis.set_xlim(-0.48, 2.48)
        axis.set_ylim(*padded_limits(panel))
        axis.yaxis.set_major_locator(MaxNLocator(nbins=5))
        axis.grid(axis='y', alpha=0.20, linewidth=0.55)
        axis.set_axisbelow(True)
        for spine in axis.spines.values():
            spine.set_color('0.35')
            spine.set_linewidth(0.65)

    axes[0].set_ylabel(r'IGD $\downarrow$' if metric == 'IGD' else r'HV $\uparrow$')
    legend_handles = [
        Line2D(
            [0], [0], linestyle='none', marker=METHOD_MARKERS[method],
            markerfacecolor=METHOD_COLORS[method], markeredgecolor='white',
            markeredgewidth=0.45,
            markersize=6.0 if method == 'C-NBI' else 5.2,
            label='NBI original ($m=4$)' if method == 'NBI original' else method,
        )
        for method in METHOD_ORDER
    ]
    figure.legend(
        handles=legend_handles, loc='lower center',
        bbox_to_anchor=(0.5, 0.045), ncol=5, frameon=False,
        columnspacing=1.15, handletextpad=0.35,
    )

    png_path = OUT_DIR / f'{output_stem}.png'
    pdf_path = OUT_DIR / f'{output_stem}.pdf'
    figure.savefig(png_path, dpi=300)
    figure.savefig(pdf_path, dpi=300)
    plt.close(figure)
    return png_path, pdf_path

outputs = {}
for metric, metric_stem in [('IGD', 'igd'), ('HV', 'hv')]:
    for version, version_stem in [('complete', 'completa'), ('equalized', 'equalizada')]:
        stem = f'fig_{metric_stem}_{version_stem}'
        outputs[(metric, version)] = plot_median_iqr(
            long_data, metric, version, stem
        )
outputs

In [ ]:
metadata = {
    'source': SOURCE_PATH.relative_to(ROOT).as_posix(),
    'input_is_recomputed': False,
    'long_format_columns': ['scenario', 'm', 'correlation', 'method', 'seed', 'metric', 'value', 'version'],
    'versions': VERSION_ORDER,
    'metrics': ['IGD', 'HV'],
    'methods': METHOD_ORDER,
    'method_colors': METHOD_COLORS,
    'method_markers': METHOD_MARKERS,
    'replicates_per_interval': 10,
    'interval': 'Q1-Q3',
    'center': 'median',
    'nbi_original_rule': 'included only when m=4',
    'panel_y_scales': 'independent',
    'publication_width_cm': FIGURE_WIDTH_CM,
}
metadata_path = OUT_DIR / 'median_iqr_figures_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

artifacts = [long_path, summary_path, metadata_path]
for pair in outputs.values():
    artifacts.extend(pair)
for artifact in artifacts:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Arquivos gerados:')
for artifact in artifacts:
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura das figuras

O marcador representa a mediana das dez repetições e a barra vertical representa o intervalo interquartil. As escalas verticais são independentes entre os painéis para acomodar as diferenças de magnitude entre $m=4$, $m=6$ e $m=12$. A comparação entre frentes completas e cardinalidade equalizada permanece separada em figuras distintas.